# Session 2 Exercise: Parameter and Rule Reform

Goal: compare baseline, parameter reform, and combined parameter+rule reform for income tax.

## Task Description

Implement the following policy changes and compare tax outcomes:
1. Use the high-income couple persona and extend the household to five children (so the cap can bind).
2. Increase `einkommensteuer__parameter_kinderfreibetrag['sächliches_existenzminimum']` by **800 EUR**.
3. Change the rule of `kinderfreibetrag_y` to: use at most four child-allowance units (`min(anzahl_kinderfreibeträge, 4)`).
4. Produce one comparison table with baseline, parameter-only reform, and combined reform.

In [ ]:
# Command for installing gettsim (only use on Colab)

# !pip install gettsim
# !pip install git+https://github.com/ttsim-dev/gettsim-personas.git

In [10]:
import numpy as np
import pandas as pd
import dags.tree as dt

from gettsim import InputData, MainTarget, TTTargets, copy_environment, main
from gettsim.tt import DictParam, policy_function

In [11]:
POLICY_DATE = "2025-01-01"

## 1) Baseline setup

In [12]:
from gettsim_personas import einkommensteuer_sozialabgaben

persona = einkommensteuer_sozialabgaben.Couple1Child(policy_date_str=POLICY_DATE)

# Raise wages so tax effects are easier to see.
persona_high_income = persona.upsert_input_data(
    {"einnahmen": {"bruttolohn_m": np.array([9000.0, 7000.0, 0.0])}}
)

print(persona.description)

Persona to compute income taxes and social insurance contributions. Jointly
        taxed married couple with one child. All transfers are set to zero; don't use
        this persona for low- to mid-income households, as they may be eligible for
        (means-tested) transfers.


## 2) Expand household with additional children

To make a cap on `anzahl_kinderfreibeträge` visible, we append copies of the child row.

In [13]:
df_persona = pd.DataFrame(dt.flatten_to_qnames(persona_high_income.input_data_tree))

child_row = df_persona.loc[df_persona["p_id"] == 2].copy()
new_children = []
for new_id in [3, 4, 5, 6]:
    r = child_row.copy()
    r["p_id"] = new_id
    new_children.append(r)

df_extended = pd.concat([df_persona] + new_children, ignore_index=True)
df_extended

,einnahmen__bruttolohn_m,einnahmen__kapitalerträge_y,alter,arbeitsstunden_w,behinderungsgrad,einkommensteuer__abzüge__beitrag_private_rentenversicherung_m,einkommensteuer__abzüge__kinderbetreuungskosten_m,einkommensteuer__abzüge__p_id_kinderbetreuungskostenträger,einkommensteuer__einkünfte__aus_forst_und_landwirtschaft__betrag_m,einkommensteuer__einkünfte__aus_gewerbebetrieb__betrag_m,...,familie__p_id_elternteil_1,familie__p_id_elternteil_2,geburtsjahr,hh_id,kindergeld__in_ausbildung,kindergeld__p_id_empfänger,p_id,sozialversicherung__kranken__beitrag__bemessungsgrundlage_rente_m,sozialversicherung__kranken__beitrag__privat_versichert,sozialversicherung__pflege__beitrag__hat_kinder
0,9000.0,500.0,30,39,0,0,0.0,-1,0,0,...,-1,-1,1995,0,False,-1,0,0,False,True
1,7000.0,0.0,30,39,0,0,0.0,-1,0,0,...,-1,-1,1995,0,False,-1,1,0,False,True
2,0.0,0.0,10,0,0,0,100.0,0,0,0,...,0,1,2015,0,False,0,2,0,False,False
3,0.0,0.0,10,0,0,0,100.0,0,0,0,...,0,1,2015,0,False,0,3,0,False,False
4,0.0,0.0,10,0,0,0,100.0,0,0,0,...,0,1,2015,0,False,0,4,0,False,False
5,0.0,0.0,10,0,0,0,100.0,0,0,0,...,0,1,2015,0,False,0,5,0,False,False
6,0.0,0.0,10,0,0,0,100.0,0,0,0,...,0,1,2015,0,False,0,6,0,False,False


In [14]:
extended_tree = dt.unflatten_from_qnames(
    {col: df_extended[col].to_numpy() for col in df_extended.columns}
)

In [15]:
# Request only one target to keep the reform effect easy to interpret.
tt_targets = TTTargets(tree={"einkommensteuer": {"betrag_y_sn": None}})

# Build the baseline policy environment for the selected date.
status_quo_environment = main(
    main_target=MainTarget.policy_environment,
    policy_date_str=POLICY_DATE,
    include_warn_nodes=False,
)

# Run baseline with extended household input data.
baseline = main(
    main_target=MainTarget.results.df_with_nested_columns,
    policy_date_str=POLICY_DATE,
    input_data=InputData.tree(tree=extended_tree),
    tt_targets=tt_targets,
    include_warn_nodes=False,
)

baseline

,einkommensteuer
,betrag_y_sn
p_id,
0,39941.0
1,39941.0
2,0.0
3,0.0
4,0.0
5,0.0
6,0.0


## 3) Parameter reform

Increase one child-allowance parameter (`sächliches_existenzminimum`).

In [16]:
# Copy first: never edit status-quo environment in place.
param_reform_environment = copy_environment(status_quo_environment)

# Inspect and modify the child allowance parameter dictionary.
old_kfb = status_quo_environment["einkommensteuer"]["parameter_kinderfreibetrag"].value
new_kfb = dict(old_kfb)
new_kfb["sächliches_existenzminimum"] = old_kfb["sächliches_existenzminimum"] + 800

param_reform_environment["einkommensteuer"]["parameter_kinderfreibetrag"] = DictParam(value=new_kfb)

# Run model with parameter reform only.
param_reform = main(
    main_target=MainTarget.results.df_with_nested_columns,
    policy_date_str=POLICY_DATE,
    input_data=InputData.tree(tree=extended_tree),
    tt_targets=tt_targets,
    policy_environment=param_reform_environment,
    include_warn_nodes=False,
)

param_reform

,einkommensteuer
,betrag_y_sn
p_id,
0,37033.0
1,37033.0
2,0.0
3,0.0
4,0.0
5,0.0
6,0.0


## 4) Rule reform

Cap `anzahl_kinderfreibeträge` at 4 in `kinderfreibetrag_y`.

In [17]:
# Start combined reform from parameter-reformed environment.
combined_reform_environment = copy_environment(param_reform_environment)

@policy_function()
def kinderfreibetrag_y(
    anzahl_kinderfreibeträge: int,
    kinderfreibetrag_pro_kind_y: float,
) -> float:
    capped_units = min(anzahl_kinderfreibeträge, 4)
    out = kinderfreibetrag_pro_kind_y * capped_units
    return out

# Replace original function with capped version.
combined_reform_environment["einkommensteuer"]["kinderfreibetrag_y"] = kinderfreibetrag_y

# Run model with parameter + rule reform together.
combined_reform = main(
    main_target=MainTarget.results.df_with_nested_columns,
    policy_date_str=POLICY_DATE,
    input_data=InputData.tree(tree=extended_tree),
    tt_targets=tt_targets,
    policy_environment=combined_reform_environment,
    include_warn_nodes=False,
)

combined_reform

,einkommensteuer
,betrag_y_sn
p_id,
0,41136.0
1,41136.0
2,0.0
3,0.0
4,0.0
5,0.0
6,0.0


## 5) Comparison

In [18]:
# Side-by-side comparison of tax outcomes.
comparison = pd.DataFrame({
    "baseline_tax": baseline[("einkommensteuer", "betrag_y_sn")],
    "param_reform_tax": param_reform[("einkommensteuer", "betrag_y_sn")],
    "combined_reform_tax": combined_reform[("einkommensteuer", "betrag_y_sn")],
})
comparison["delta_param"] = comparison["param_reform_tax"] - comparison["baseline_tax"]
comparison["delta_combined"] = comparison["combined_reform_tax"] - comparison["baseline_tax"]
comparison

,baseline_tax,param_reform_tax,combined_reform_tax,delta_param,delta_combined
p_id,,,,,
0,39941.0,37033.0,41136.0,-2908.0,1195.0
1,39941.0,37033.0,41136.0,-2908.0,1195.0
2,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0
6,0.0,0.0,0.0,0.0,0.0
